加载model

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. 必须先实例化你之前定义的那个网络结构（空壳）
class HardwareFriendlyCNN(nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        # 只使用基础算子，方便后续转纯 C 语言和 NPU 硬件实现
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # CIFAR-10 尺寸: 32x32 -> pool -> 16x16 -> pool -> 8x8
        self.fc = nn.Linear(32 * 8 * 8, 10) 
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model = HardwareFriendlyCNN()

# 2. 读取 .pth 文件中的权重字典，并加载到模型中
# (加上 weights_only=True 是 PyTorch 新版本的安全规范)
model.load_state_dict(torch.load('./my_hardware_cnn_fp32_69p.pth', weights_only=True))

# 3. 极其重要：将模型切入评估/推理模式
# 这会关闭 Dropout 和 BatchNorm 的动态更新（虽然我们这个极简模型没有这些，但这是标准规范）
model.eval() 

print("模型加载成功！")

模型加载成功！


加载my_flatten函数

In [2]:
# ==========================================
# 0. 定义手动展平函数
# ==========================================

def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    假设输入 x 的形状为 (Batch, Channels, Height, Width)
    """
    B, C, H, W = x.shape
    
    # 准备一个普通的 Python 列表，充当我们的“一维连续内存”
    flattened_memory = []
    
    # 严格按照行优先的多重循环读取数据
    # 最外层是 Batch，然后是 Channel，接着是 Height，最内层是连续的 Width
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    # 每次读取一个具体的数值，追加到一维内存中
                    # 这个遍历顺序完美等价于我们之前推导的 C 语言一维索引计算公式：
                    # Index = c * (H * W) + h * W + w
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
                    
    # 将我们手工铺平的一维列表重新包装成 PyTorch 张量
    # 形状为 (B, C * H * W)，在我们的例子中就是 (1, 2048)
    # x.dtype 和 x.device 确保数据类型和所在硬件（CPU/GPU）与原输入保持一致
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

fp32模型正确性测试（数据流动方向）

In [6]:
def main():
    # 检测是否能用 GPU 加速训练
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ==========================================
    # 2. 准备 CIFAR-10 数据集 (自动下载)
    # ==========================================
    transform = transforms.Compose([
        transforms.ToTensor(), # 将图片转为 [0.0, 1.0] 的浮点数 Tensor
        # 这里为了简化硬件 C 代码的预处理，我们省略了 Normalize(均值方差归一化)
    ])

    print("Loading datasets...")
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

    # ==========================================
    # 3. 训练模型 (这一步将生成属于你的开源权重)
    # ==========================================
    model.to(device)
    
    # ==========================================
    # 4. 在测试集上进行全局准确率验证
    # ==========================================
    print("Testing FP32 model accuracy on 10000 test images...")
    model.eval() # 切换到推理模式
    correct = 0
    total = 0
    
    with torch.no_grad(): # 推理时不需要计算梯度
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1) # 获取得分最大的类别
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"FP32 Model Accuracy on the 10000 test images: {accuracy:.2f} %")

if __name__ == '__main__':
    main()

Using device: cuda
Loading datasets...
Files already downloaded and verified
Files already downloaded and verified
Testing FP32 model accuracy on 10000 test images...
FP32 Model Accuracy on the 10000 test images: 69.33 %


8bit非对称量化

In [4]:
import torch
import numpy as np

# 假设你已经定义了上一节的 HardwareFriendlyCNN 类
# from cifar10_fp32_baseline import HardwareFriendlyCNN

def calculate_scale_zp(tensor, q_min=-128, q_max=127):
    """计算张量的 Scale 和 Zero Point"""
    r_min = tensor.min().item()
    r_max = tensor.max().item()
    
    # 避免除以 0
    if r_min == r_max:
        return 1.0, 0

    scale = (r_max - r_min) / (q_max - q_min)
    zero_point = round(q_min - (r_min / scale))
    
    # 限制 Zero Point 在 INT8 范围内
    zero_point = max(q_min, min(q_max, zero_point))
    return scale, zero_point

def quantize_tensor(tensor, scale, zero_point, q_min=-128, q_max=127):
    """使用算出的 Scale 和 ZP 将 FP32 转换为 INT8"""
    q_tensor = torch.round(tensor / scale + zero_point)
    q_tensor = torch.clamp(q_tensor, q_min, q_max)
    return q_tensor.to(torch.int8)

def export_c_array(tensor, name, file_handle, is_int32=False):
    """将量化后的张量导出为 C 语言数组"""
    flat_data = tensor.flatten().numpy()
    c_type = "int32_t" if is_int32 else "int8_t"
    
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ")
    for i, val in enumerate(flat_data):
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def main():
    # # 1. 加载你用 4070Ti Super 秒速训练好的 FP32 模型
    # model.load_state_dict(torch.load('my_hardware_cnn_fp32.pth', weights_only=True))
    # model.eval()

    with open("npu_weights_int8.h", "w") as f:
        f.write("#ifndef NPU_WEIGHTS_INT8_H\n#define NPU_WEIGHTS_INT8_H\n")
        f.write("#include <stdint.h>\n\n")

        # 2. 提取第一层卷积的权重 (以 conv1 为例)
        fp32_weight = model.conv1.weight.data
        fp32_bias = model.conv1.bias.data

        # 3. 计算权重的 Scale 和 ZP
        w_scale, w_zp = calculate_scale_zp(fp32_weight)
        
        # 4. 执行量化，变成纯 INT8
        int8_weight = quantize_tensor(fp32_weight, w_scale, w_zp)

        # 【硬件设计核心提醒】
        # 脉动阵列通常是 INT8 * INT8，累加器是 INT32。
        # 所以 Bias 通常量化为 INT32，它的 Scale 是 (输入特征图的Scale * 权重的Scale)
        # 这里为了演示，我们假设输入特征图的 Scale=1.0（实际工程中你需要跑几张图统计输入的 min/max）
        input_scale = 1.0 
        bias_scale = input_scale * w_scale
        int32_bias = torch.round(fp32_bias / bias_scale).to(torch.int32)

        # 5. 写入 C 头文件
        f.write(f"// Conv1 Quantization Params: Scale={w_scale:.6f}, ZeroPoint={w_zp}\n")
        f.write(f"// Note: NPU hardware needs these params to requantize the INT32 accumulator back to INT8\n")
        export_c_array(int8_weight, "conv1_weight_int8", f)
        export_c_array(int32_bias, "conv1_bias_int32", f, is_int32=True)
        
        f.write("#endif // NPU_WEIGHTS_INT8_H\n")

    print("INT8 量化完成！已导出为 npu_weights_int8.h 供裸机 C 语言调用。")

if __name__ == '__main__':
    main()

INT8 量化完成！已导出为 npu_weights_int8.h 供裸机 C 语言调用。
